
* Uso de la librería de Google GenAI del modelo gratuito gemini-3.8-flash
*   Obtener la API-KEY de Google AI Studio: [Enlace](https://ai.google.dev/gemini-api/docs/api-key)
*   Almacenar la clave en la sección de secretos como GOOGLE_GENAI_API_KEY.



In [ ]:
import time

from google import genai
from google.colab import userdata
import pandas as pd

## Inicialización de constantes

In [ ]:
TIEMPO_ENTRE_REINTENTOS = 30 # Segundos

## Inicialización del cliente de Google GenAI

In [ ]:
# Recuperar la clave desde los secretos de Colab
google_api_key = userdata.get('GOOGLE_GENAI_API_KEY')

# Inicializar el cliente de genai
client = genai.Client(api_key=google_api_key)

In [ ]:
from google.genai.errors import ServerError

def get_valoracion_vivienda(descripcion: str) -> int:
  try:
    # Generar el contenido usando el modelo gratuito
    response = client.models.generate_content(
      model='gemini-3.8-flash',
      contents='Valora este inmueble. Proporciona como respuesta únicamente su valor en euros.' + descripcion,
    )
    return int(response.text.strip().replace('.', '').replace('€', '').replace(' ', ''))
  except ServerError as e:
    if e.code == 503:
      raise Exception("El servicio de valoración no está disponible temporalmente debido a una alta demanda (Error 503).") from e
    raise e

In [ ]:
datos = {
    "propietarios":['Propietario 1', 'Propietario 2', 'Propietario 3', 'Propietario 4'],
    "viviendas":[
        'Vivienda unifamiliar de 120 metros cuadrados en Tejeda de Tiétar',
        'Apartamento de 2 dormitorios en Benidorm. Primera línea de playa',
        'Piso de 3 dormitorios y 90 metros cuadrados en el Delicias, Zaragoza',
        'Prado de pastos en Asturias de una hectárea'],
}

df = pd.DataFrame(datos)
display(df)

,propietarios,viviendas
0,Propietario 1,Vivienda unifamiliar de 120 metros cuadrados e...
1,Propietario 2,Apartamento de 2 dormitorios en Benidorm. Prim...
2,Propietario 3,Piso de 3 dormitorios y 90 metros cuadrados en...
3,Propietario 4,Prado de pastos en Asturias de una hectárea


In [ ]:
while True:
    try:
        df['valoracion'] = df['viviendas'].map(get_valoracion_vivienda)
        display(df)
        break  # Si termina correctamente, sale del bucle

    except Exception as e:
        print(f"Error: {e}")
        print(f"Reintentando en {TIEMPO_ENTRE_REINTENTOS} segundos...")
        time.sleep(TIEMPO_ENTRE_REINTENTOS)

Error: El servicio de valoración no está disponible temporalmente debido a una alta demanda (Error 503).
Reintentando en 30 segundos...
Error: El servicio de valoración no está disponible temporalmente debido a una alta demanda (Error 503).
Reintentando en 30 segundos...
Error: El servicio de valoración no está disponible temporalmente debido a una alta demanda (Error 503).
Reintentando en 30 segundos...
Error: El servicio de valoración no está disponible temporalmente debido a una alta demanda (Error 503).
Reintentando en 30 segundos...
Error: El servicio de valoración no está disponible temporalmente debido a una alta demanda (Error 503).
Reintentando en 30 segundos...


,propietarios,viviendas,valoracion
0,Propietario 1,Vivienda unifamiliar de 120 metros cuadrados e...,80000
1,Propietario 2,Apartamento de 2 dormitorios en Benidorm. Prim...,350000
2,Propietario 3,Piso de 3 dormitorios y 90 metros cuadrados en...,135000
3,Propietario 4,Prado de pastos en Asturias de una hectárea,12500
